In [1]:
import agents.encounter_designer as ed
import agents.enemy as e
import agents.npc as npc

import json
from pydantic import BaseModel, Field
from typing import List, Type, Optional

from langchain.agents import AgentExecutor, create_openai_tools_agent
from langchain.chat_models import ChatOpenAI
from langchain.callbacks.manager import (
    AsyncCallbackManagerForToolRun,
    CallbackManagerForToolRun,
)
from langchain_core.agents import AgentFinish
from langchain_core.messages import SystemMessage
from langchain_core.prompts import ChatPromptTemplate, HumanMessagePromptTemplate, MessagesPlaceholder
from langchain_core.tools import BaseTool, Tool

/home/vscode/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
cura = npc.NPC(name="Captain Cura")
gually = npc.NPC(name="Gualluwyrm the Awesome")

/home/vscode/.local/lib/python3.10/site-packages/langchain_core/_api/deprecation.py:117: LangChainDeprecationWarning: The class `langchain_community.chat_models.openai.ChatOpenAI` was deprecated in langchain-community 0.0.10 and will be removed in 0.2.0. An updated version of the class exists in the langchain-openai package and should be used instead. To use it run `pip install -U langchain-openai` and import as `from langchain_openai import ChatOpenAI`.
  warn_deprecated(


In [3]:
system_prompt = """
You are a D&D dungeon master and a powerful storyteller. You are a student of Joseph Campbell and you know how to structure the beats of an adventure.

You will design a story involving 2 to 4 thematic encounters.
The setting and party for the adventure is: {input}

You have the following tools:
EncounterDesigner - use this to design encounters appropriate for the players' levels, desired difficulty, and setting (In keeping with Joseph Campbell's insights, the difficulty should rise until we hit an exciting climax, but should never be too difficult for the players to overcome.)
StoryFormatter - use this to structure the story in the desired output format

Output Format: return an array of the encounters in the adventure. Use `StoryFormatter` to structure them into the desired format.
{agent_scratchpad}
"""

In [4]:
system_prompt = """
You are a D&D dungeon master and a powerful storyteller. You are a student of Joseph Campbell and you know how to structure the beats of an adventure.

You will design a story involving 2 to 4 thematic story hooks. In keeping with the teachings of Joseph Campbell, the drama should increase as the story continues.
The setting and party for the adventure is: {input}

You have the following tools:
StoryFormatter - use this to structure the story in the desired output format

Output Format: return an array of the story hooks in the adventure. Use `StoryFormatter` to structure them into the desired format.
{agent_scratchpad}
"""

In [5]:
system_prompt = """
You are a D&D dungeon master and a powerful storyteller. You are a student of Joseph Campbell and you know how to structure the beats of an adventure.

You will design a story involving 3 thematic story hooks. In keeping with Joseph Campbell's insights, the drama should increase as the story continues.
The setting and party for the adventure is: {input}

After you have come up with the story hooks, use them to generate encounters for the party to overcome.

You have the following tools:
EncounterDesigner - use this to design encounters appropriate for the players' levels, desired difficulty, setting, and story hook. IMPORTANT: You MUST include the desired difficulty and the story hook!
StoryFormatter - use this to structure the encounters in the desired output format

Output Format: return the encounters in the adventure. IMPORTANT: You MUST use `StoryFormatter` to structure them into the desired format.
{agent_scratchpad}
"""

In [6]:
class StoryHook(BaseModel):
    description: str = Field("description of the story hook or encounter")
    encounter: ed.Encounter = Field("the encounter for the story hook")

class StoryFormatterArgs(BaseModel):
    hooks: List[StoryHook] = Field("the events the players will experience throughout the story")

class StoryFormatter(BaseTool):
    name = "StoryFormatter"
    description = "format a set of encounters"
    args_schema: Type[BaseModel] = StoryFormatterArgs

    def _run(
        self, hooks: List[StoryHook],
        run_manager: Optional[CallbackManagerForToolRun] = None
    ):
        """Use the tool."""
        loggable = [s.model_dump() for s in hooks]
        return AgentFinish(return_values={"hooks": hooks}, log=json.dumps(loggable))
    
    async def _arun(
        self,
        hooks: List[str],
        run_manager: Optional[AsyncCallbackManagerForToolRun] = None,
    ) -> str:
        """Use the tool asynchronously."""
        raise NotImplementedError("does not support async")

In [7]:
tools = [
    ed.EncounterDesigner(),
    StoryFormatter()
]

In [8]:
llm = ChatOpenAI(temperature=0)
prompt = ChatPromptTemplate.from_template(template=system_prompt)
agent = create_openai_tools_agent(llm, tools, prompt)

In [9]:
executor = AgentExecutor(agent=agent, tools=tools, verbose=True, handle_parsing_errors=True)
# f u langchain I can return direct if I want to
tools[-1].return_direct = True

In [10]:
ed.EncounterDesignerArgs()

EncounterDesignerArgs(player_levels='an integer list of the players in the party', desired_difficulty='the desired difficulty', setting='the desired narrative and setting for the encounter', story_hook='the situation and events leading up to the encounter')

In [11]:
executor.invoke({"input": "setting: a bustling bazaar, party: level 1 barbarian, level 1 ranger"})



> Entering new AgentExecutor chain...

Invoking: `EncounterDesigner` with `{'player_levels': [1, 1], 'desired_difficulty': 'easy', 'setting': 'a bustling bazaar', 'story_hook': 'The adventurers stumble upon a mysterious merchant selling enchanted trinkets'}`




> Entering new AgentExecutor chain...

Invoking: `DifficultyMap` with `{'player_levels': [1, 1], 'desired_difficulty': 'Easy'}`


(0, 50)
Invoking: `StructureEncounter` with `{'enemies': [{'name': 'Enchanted Trinket Merchant', 'stats': {'armor_class': 12, 'hit_point': 5, 'speed': '30 ft', 'initiative': '+1'}, 'cr': '0', 'xp': 10, 'other_info': 'A mysterious merchant selling enchanted trinkets.'}]}`


return_values={'enemies': [Enemy(name='Enchanted Trinket Merchant', stats=StatBlock(armor_class=12, hit_point=5, speed='30 ft', initiative='+1', strength="The enemy's strength (STR), should be of the form [[Value]] ([[Modifier]])", dexterity="The enemy's dexterity (DEX), should be of the form [[Value]] ([[Modifier]])", constituti

/home/vscode/.local/lib/python3.10/site-packages/pydantic/main.py:314: UserWarning: Pydantic serializer warnings:
  Expected `list[Action]` but got `str` - serialized value may not be as expected
  return self.__pydantic_serializer__.to_python(
/home/vscode/.local/lib/python3.10/site-packages/pydantic/main.py:358: UserWarning: Pydantic serializer warnings:
  Expected `list[Action]` but got `str` - serialized value may not be as expected
  return self.__pydantic_serializer__.to_json(
/home/vscode/.local/lib/python3.10/site-packages/pydantic/main.py:314: UserWarning: Pydantic serializer warnings:
  Expected `list[Action]` but got `str` - serialized value may not be as expected
  return self.__pydantic_serializer__.to_python(
/home/vscode/.local/lib/python3.10/site-packages/pydantic/main.py:358: UserWarning: Pydantic serializer warnings:
  Expected `list[Action]` but got `str` - serialized value may not be as expected
  return self.__pydantic_serializer__.to_json(


### Encounter: Mysterious Merchant in the Bazaar

#### Enemies:
1. **Enchanted Trinket Merchant**
   - **CR:** 0
   - **XP:** 10
   - **Stats:**
     - **Armor Class:** 12
     - **Hit Points:** 5
     - **Speed:** 30 ft
     - **Initiative:** +1
   - **Description:** A mysterious merchant selling enchanted trinkets.

2. **Enchanted Trinket**
   - **CR:** 0
   - **XP:** 10
   - **Stats:**
     - **Armor Class:** 10
     - **Hit Points:** 1
     - **Speed:** 0 ft
     - **Initiative:** +0
   - **Description:** A magical trinket that can animate and assist the merchant.

The adventurers stumble upon a bustling bazaar and find a mysterious merchant selling enchanted trinkets. The enchanted trinkets come to life to assist the merchant in unexpected ways. This encounter provides a light-hearted and magical experience for the players.

> Finished chain.
### Encounter: Mysterious Merchant in the Bazaar

#### Enemies:
1. **Enchanted Trinket Merchant**
   - **CR:** 0
   - **XP:** 10
   - **Stat

Exception: ('Expected JSON response, got ', {'instructions': 'Design an encounter of easy difficulty for players of levels [1, 1]. a bustling bazaar. The adventurers stumble upon a mysterious merchant selling enchanted trinkets', 'output': '### Encounter: Mysterious Merchant in the Bazaar\n\n#### Enemies:\n1. **Enchanted Trinket Merchant**\n   - **CR:** 0\n   - **XP:** 10\n   - **Stats:**\n     - **Armor Class:** 12\n     - **Hit Points:** 5\n     - **Speed:** 30 ft\n     - **Initiative:** +1\n   - **Description:** A mysterious merchant selling enchanted trinkets.\n\n2. **Enchanted Trinket**\n   - **CR:** 0\n   - **XP:** 10\n   - **Stats:**\n     - **Armor Class:** 10\n     - **Hit Points:** 1\n     - **Speed:** 0 ft\n     - **Initiative:** +0\n   - **Description:** A magical trinket that can animate and assist the merchant.\n\nThe adventurers stumble upon a bustling bazaar and find a mysterious merchant selling enchanted trinkets. The enchanted trinkets come to life to assist the merc

In [ ]:
story = _

In [ ]:
story["output"].return_values["hooks"]

[StoryHook(description='The party is exploring the bazaar when they notice a group of shady figures exchanging a mysterious package.', encounter=Encounter(enemies=[Enemy(name='Shady Figure', stats=StatBlock(armor_class=12, hit_point=16, speed='30 ft.', initiative='+2', strength='10', dexterity='14', constitution='12', intelligence='10', wisdom='12', charisma='14'), cr='1/2', xp=100, actions=[Action(name='Dagger', description='Melee Weapon Attack: +4 to hit, reach 5 ft., one target. Hit: 4 (1d4 + 2) piercing damage.')], other_info='The shady figure is a nimble and stealthy individual, skilled in close combat.')])),
 StoryHook(description='The party is ambushed by a group of guards who suspect them of being involved with the shady figures.', encounter=Encounter(enemies=[Enemy(name='Shady Figure', stats=StatBlock(armor_class=12, hit_point=16, speed='30 ft.', initiative='+2', strength='10', dexterity='14', constitution='12', intelligence='10', wisdom='12', charisma='14'), cr='1/2', xp=100,